# π0.5-LIBERO × PARC 2026 提出環境

`feature/experiment` の `MyPolicy` をGPUでスモークし、公開4タスクで評価して、外部通信なしで動く `pi05_submission.zip` を作るノートブックです。既存のSmolVLAノートブックとは環境を分離します。

- ランタイム: GPU（L4 / A100推奨）
- PaliGemmaの利用条件にHugging Face上で同意してください
- W&BはLoRA学習時だけ任意で使用し、評価時は無効化します
- 公式LIBEROデモによるAction Expert LoRAを含みます。公開評価の観測・結果は学習に使いません


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"

REPO_URL = "https://github.com/KosukeKomeya/PARC2026_pre.git"
BRANCH = "feature/experiment"
REPO_DIR = Path("/content/PARC2026_pre")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
    check=True,
)
%cd /content/PARC2026_pre


In [ ]:
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError("ColabのランタイムをGPUに変更してください")

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print("VRAM GiB:", round(props.total_memory / 2**30, 1))
subprocess.run(["nvidia-smi"], check=True)


## Hugging Faceへログイン

`google/paligemma-3b-pt-224` のページで利用条件へ同意してから実行します。トークンをコードやGitHubへ書かないでください。


In [ ]:
%pip install -q "huggingface-hub>=0.34.2,<0.36.0"
from huggingface_hub import notebook_login
notebook_login()


## 1. 固定バージョンの環境と重みを準備

Python 3.10の専用venvを作り、固定コミットのLeRobot/Transformers、固定revisionのπ0.5-LIBERO重み、tokenizerを取得します。初回は時間がかかります。


In [ ]:
!python examples/pi05_parc_colab_setup.py


## 2. 実モデルでスモークテスト

モデルのロード、最初の重い推論、action queueから返す軽い推論を計測します。重い推論が10秒以上なら提出前にGPU・推論step数を見直してください。


In [ ]:
!python examples/pi05_parc_colab_setup.py --reuse-runtime --skip-download --smoke


## 3. Action ExpertへLoRA追加学習

既存π0.5の画像・言語部分は固定し、行動を作る別Transformer（Action Expert）の `q_proj` / `v_proj` と行動・時刻projectionだけへLoRAを挿します。公式LIBERO全40タスクのデモをタスクごと同数にし、エピソード丸ごとで学習用／検証用へ分割します。軽い色変化は使いますが、左右反転や位置を変える幾何変換は使いません。

最大3,000 steps、rank 16です。A100はbatch 4、L4はbatch 2を自動選択し、5-step事前試験がCUDA OOMになった場合だけ半減します。250 stepsごとのLoRA再開checkpointはDriveへ原子的に退避し、ランタイム切断後は最新stepから再開します。10 stepsごとにloss・学習率・gradient normを表示し、W&Bにも記録できます。


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PI05_PYTHON = Path("/content/pi05_py310/bin/python")
RUN_LORA_TRAINING = True
USE_WANDB = True  # 登録済みならTrue。使わない場合はFalse
COPY_MERGED_MODEL_TO_DRIVE = False  # ZIPとcheckpointで復元可能。Driveに十分な空きがある場合だけTrue
TRAIN_STEPS = 3000
SAVE_FREQ = 250
PREFLIGHT_STEPS = 5
LORA_RANK = 16
TRAIN_BATCH_SIZE = 0  # 0: A100=4 / L4=2を自動選択し、OOM時だけ半減
DATASET_REPO = "lerobot/libero"
DATASET_REVISION = "a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4"
BASE_MODEL_DIR = REPO_DIR / "submission_template/model_weights/pi05_libero_finetuned_v044"
TRAIN_ROOT = Path("/content/pi05_action_expert_lora_full40")
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/PARC2026/pi05_action_expert_lora_full40")
SPLIT_MANIFEST = TRAIN_ROOT / "data_split.json"
TRAIN_OUTPUT_DIR = TRAIN_ROOT / "training"
DRIVE_CHECKPOINT_DIR = DRIVE_RUN_ROOT / "checkpoints"
CHECKPOINT_SELECTION = TRAIN_ROOT / "checkpoint_selection.json"
CHECKPOINT_VALIDATIONS = DRIVE_RUN_ROOT / "checkpoint_validations"
MERGED_MODEL_DIR = TRAIN_ROOT / "merged_model"
LORA_VALIDATION = TRAIN_ROOT / "merged_validation.json"
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("LoRA training enabled:", RUN_LORA_TRAINING)
print("steps / rank / requested batch:", TRAIN_STEPS, LORA_RANK, TRAIN_BATCH_SIZE)
print("Drive recovery directory:", DRIVE_RUN_ROOT)


In [ ]:
if RUN_LORA_TRAINING:
    TRAIN_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            str(PI05_PYTHON), "examples/pi05_action_expert_lora.py", "prepare",
            "--dataset-repo", DATASET_REPO,
            "--dataset-revision", DATASET_REVISION,
            "--output", str(SPLIT_MANIFEST),
            "--validation-per-task", "2",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    shutil.copy2(SPLIT_MANIFEST, DRIVE_RUN_ROOT / "data_split.json")
else:
    print("LoRA学習をスキップします。既存のπ0.5重みで提出を作ります。")


### 分割内容を確認

全40タスクから同数を選び、各タスク2エピソードを検証専用にします。公開4タスク評価の観測や成否は、この分割にも学習にも使用しません。


In [ ]:
if RUN_LORA_TRAINING:
    split_info = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
    print("tasks:", len(split_info["task_counts"]))
    print("train episodes:", len(split_info["train_episodes"]))
    print("validation episodes:", len(split_info["validation_episodes"]))
    assert len(split_info["task_counts"]) == 40
    assert set(split_info["train_episodes"]).isdisjoint(split_info["validation_episodes"])


### LoRA学習を実行

W&Bを使う場合、未ログインなら安全な対話プロンプトでAPI keyを入力します。キーをセルやGitHubへ書かないでください。学習中は `loss`, `grdn`, `lr`, `step`, `eta` が10 stepsごとに表示されます。


In [ ]:
if RUN_LORA_TRAINING:
    if USE_WANDB:
        subprocess.run([str(PI05_PYTHON.parent / "wandb"), "login"], check=True)
    train_command = [
        str(PI05_PYTHON), "examples/pi05_action_expert_lora.py", "train",
        "--manifest", str(SPLIT_MANIFEST),
        "--base-model", str(BASE_MODEL_DIR),
        "--output-dir", str(TRAIN_OUTPUT_DIR),
        "--steps", str(TRAIN_STEPS),
        "--save-freq", str(SAVE_FREQ),
        "--preflight-steps", str(PREFLIGHT_STEPS),
        "--batch-size", str(TRAIN_BATCH_SIZE),
        "--lora-rank", str(LORA_RANK),
        "--backup-dir", str(DRIVE_CHECKPOINT_DIR),
        *( ["--wandb"] if USE_WANDB else [] ),
    ]
    subprocess.run(train_command, cwd=REPO_DIR, check=True)


### checkpointを検証データで選び、通常重みへ統合

500 stepsごとの候補を、学習に使っていない同じ64サンプル・同じseedで比較します。3,000-step目を無条件採用せず、検証lossが最小のadapterだけを元のπ0.5へ統合します。選択にPARC公開評価の結果は使いません。統合後にも同条件でlossを再測定し、adapter選択時と大きく違えば停止します。


In [ ]:
if RUN_LORA_TRAINING:
    subprocess.run(
        [
            str(PI05_PYTHON), "examples/pi05_action_expert_lora.py", "select",
            "--manifest", str(SPLIT_MANIFEST),
            "--base-model", str(BASE_MODEL_DIR),
            "--checkpoints-dir", str(TRAIN_OUTPUT_DIR / "checkpoints"),
            "--output-dir", str(CHECKPOINT_VALIDATIONS),
            "--output", str(CHECKPOINT_SELECTION),
            "--candidate-every", "500",
            "--max-samples", "64",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    selection = json.loads(CHECKPOINT_SELECTION.read_text(encoding="utf-8"))
    ADAPTER_DIR = Path(selection["selected"]["adapter_dir"])
    selected_loss = float(selection["selected"]["mean_flow_matching_loss"])
    print("selected adapter:", ADAPTER_DIR)
    shutil.copy2(CHECKPOINT_SELECTION, DRIVE_RUN_ROOT / "checkpoint_selection.json")
    subprocess.run(
        [
            str(PI05_PYTHON), "examples/pi05_action_expert_lora.py", "merge",
            "--base-model", str(BASE_MODEL_DIR),
            "--adapter-dir", str(ADAPTER_DIR),
            "--output-dir", str(MERGED_MODEL_DIR),
            "--training-manifest", str(SPLIT_MANIFEST),
            "--overwrite",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    subprocess.run(
        [
            str(PI05_PYTHON), "examples/pi05_action_expert_lora.py", "validate",
            "--manifest", str(SPLIT_MANIFEST),
            "--model-dir", str(MERGED_MODEL_DIR),
            "--output", str(LORA_VALIDATION),
            "--max-samples", "64",
        ],
        cwd=REPO_DIR,
        check=True,
    )
    merged_loss = json.loads(LORA_VALIDATION.read_text(encoding="utf-8"))["mean_flow_matching_loss"]
    print(f"selected adapter loss={selected_loss:.6f}, merged loss={merged_loss:.6f}")
    if abs(merged_loss - selected_loss) > max(1e-4, selected_loss * 0.05):
        raise RuntimeError("LoRA統合前後の検証lossが一致しません。提出へ進みません。")
    shutil.copy2(LORA_VALIDATION, DRIVE_RUN_ROOT / "merged_validation.json")


## 4. オフライン提出ZIPを作成

LoRAを実行した場合だけ統合済み重みを明示的に選びます。チェックポイント、tokenizer、固定ソース、`policy_server.py`、`requirements.txt`をZIPルートへまとめ、静的validationを実行します。元モデルを使う場合は `RUN_LORA_TRAINING=False` のままです。


In [ ]:
build_command = [
    str(PI05_PYTHON), "examples/pi05_parc_colab_setup.py",
    "--reuse-runtime", "--skip-download", "--build-submission",
    *( ["--model-source", str(MERGED_MODEL_DIR)] if RUN_LORA_TRAINING else [] ),
]
subprocess.run(build_command, cwd=REPO_DIR, check=True)

def copy_file_with_progress(source: Path, destination: Path, chunk_mib: int = 16):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".partial")
    total = source.stat().st_size
    copied = 0
    next_report = 512 * 2**20
    with source.open("rb") as reader, temporary.open("wb") as writer:
        while block := reader.read(chunk_mib * 2**20):
            writer.write(block)
            copied += len(block)
            if copied >= next_report or copied == total:
                print(f"DRIVE_COPY {source.name}: {copied / 2**30:.2f}/{total / 2**30:.2f} GiB", flush=True)
                next_report += 512 * 2**20
    temporary.replace(destination)

def copy_tree_with_progress(source: Path, destination: Path):
    shutil.rmtree(destination, ignore_errors=True)
    for source_file in sorted(path for path in source.rglob("*") if path.is_file()):
        relative = source_file.relative_to(source)
        print("Drive model file:", relative)
        copy_file_with_progress(source_file, destination / relative)

submission_zip = REPO_DIR / "pi05_submission.zip"
copy_file_with_progress(submission_zip, DRIVE_RUN_ROOT / "final/pi05_submission.zip")
if RUN_LORA_TRAINING and COPY_MERGED_MODEL_TO_DRIVE:
    copy_tree_with_progress(MERGED_MODEL_DIR, DRIVE_RUN_ROOT / "final/merged_model")
print("Drive final artifacts:", DRIVE_RUN_ROOT / "final")


## 5. 公開4タスクで評価

PARC配布キットの `libero_t1`（公開4タスク）を、提出時と同じHTTPインターフェースと10秒タイムアウトで評価します。評価用のCPU環境は `venv/`、π0.5サーバーは既存の `/content/pi05_py310` GPU環境で動かし、PyTorch環境を分離します。初回の評価環境準備には10〜20分かかります。

既定はクイック確認用の1 episode/task（合計4エピソード）です。より安定した成功率を確認する場合は `EVAL_EPISODES_PER_TASK = 20` に変更してください。


In [ ]:
import csv
import os

PUBLIC_TASKS_CSV = REPO_DIR / "compe" / "t1" / "T1_TASKS.csv"
with PUBLIC_TASKS_CSV.open(encoding="utf-8", newline="") as handle:
    public_task_rows = list(csv.DictReader(handle))

PUBLIC_TASK_IDS = [row["task_id"] for row in public_task_rows]
if len(PUBLIC_TASK_IDS) != 4 or len(set(PUBLIC_TASK_IDS)) != 4:
    raise RuntimeError(f"公開タスクは4件である必要があります: {PUBLIC_TASK_IDS}")

EVAL_EPISODES_PER_TASK = 1  # 正式寄りの評価では20へ変更
EVAL_MAX_STEPS = 600
EVAL_SEED = 42
REPLAN_STEPS = 5  # 比較時は10へ変え、この評価セル以降を再実行
INFERENCE_STEPS = 10
RECORD_VIDEO = True
VIDEOS_PER_TASK = 1  # 各タスクで最初の失敗1件だけ保存

if EVAL_EPISODES_PER_TASK < 1 or EVAL_MAX_STEPS < 1:
    raise ValueError("episode数とmax stepsは1以上にしてください")

print(f"公開タスク: {len(PUBLIC_TASK_IDS)}件")
for index, row in enumerate(public_task_rows, 1):
    print(f"  {index}. {row['instruction']} ({row['task_id']})")
print(f"評価量: {len(PUBLIC_TASK_IDS) * EVAL_EPISODES_PER_TASK} episodes")


### 評価環境を準備

MuJoCoの描画ライブラリ、評価専用venv、LIBERO-plus、公開タスクassetsを準備します。`setup.sh` は `~/.libero/config.yaml` を評価環境向けに設定し、既存ファイルがあれば `.bak` へ退避します。


In [ ]:
system_packages = [
    "libosmesa6", "libgl1", "libglfw3", "libglew2.2",
    "libegl1", "libsm6", "libxext6", "libxrender1",
    "libglib2.0-0", "libmagickwand-dev", "unzip",
]
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "--no-install-recommends", *system_packages],
    check=True,
)

PI05_PYTHON = Path("/content/pi05_py310/bin/python")
if not PI05_PYTHON.is_file():
    raise FileNotFoundError("先に手順1のπ0.5環境セットアップを実行してください")

setup_env = os.environ.copy()
setup_env["PYTHON"] = str(PI05_PYTHON)
setup_env["MUJOCO_GL"] = "egl"
setup_env["MPLBACKEND"] = "Agg"
subprocess.run(["bash", "setup.sh"], cwd=REPO_DIR, env=setup_env, check=True)

EVAL_PYTHON = REPO_DIR / "venv" / "bin" / "python"
if not EVAL_PYTHON.is_file():
    raise FileNotFoundError(EVAL_PYTHON)
print("評価Python:", EVAL_PYTHON)


### π0.5サーバーを起動して4タスクを実行

サーバーの起動確認後に4タスクを順番に評価し、成功率、collision rate、平均episode時間を表示します。各 `/act` と `/reset` には本番同様10秒の上限が適用されます。結果JSONとサーバーログは `results/pi05_public_eval/` に保存します。


In [ ]:
import json
import shutil
import socket
import time
import urllib.request

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_probe:
    port_probe.bind(("127.0.0.1", 0))
    SERVER_PORT = port_probe.getsockname()[1]
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"
RESULTS_DIR = REPO_DIR / "results" / f"pi05_public_eval_replan_{REPLAN_STEPS}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR = RESULTS_DIR / "videos"
if RECORD_VIDEO:
    shutil.rmtree(VIDEO_DIR, ignore_errors=True)
server_log_path = RESULTS_DIR / "policy_server.log"
evaluation_log_path = RESULTS_DIR / "evaluation.log"
public_eval_result_path = RESULTS_DIR / f"server_{SERVER_PORT}.json"
if public_eval_result_path.exists():
    public_eval_result_path.unlink()

server_env = os.environ.copy()
server_env.update(
    {
        "PI05_MODEL_DIR": str(REPO_DIR / "submission_template/model_weights/pi05_libero_finetuned_v044"),
        "PI05_TOKENIZER_DIR": str(REPO_DIR / "submission_template/model_weights/paligemma-3b-pt-224"),
        "PI05_DEVICE": "cuda",
        "PI05_REPLAN_STEPS": str(REPLAN_STEPS),
        "PI05_INFERENCE_STEPS": str(INFERENCE_STEPS),
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "TOKENIZERS_PARALLELISM": "false",
    }
)
server_env["PYTHONPATH"] = os.pathsep.join(
    [
        "/content/pi05_runtime/lerobot_v044/src",
        "/content/pi05_runtime/transformers_lerobot_openpi/src",
        server_env.get("PYTHONPATH", ""),
    ]
)

eval_env = os.environ.copy()
eval_env.update(
    {
        "MUJOCO_GL": "egl",
        "MPLBACKEND": "Agg",
        "LIBERO_ROOT": str(REPO_DIR / "LIBERO-plus"),
        "PYTHONUNBUFFERED": "1",
    }
)
eval_env["PYTHONPATH"] = os.pathsep.join(
    [str(REPO_DIR / "LIBERO-plus"), str(REPO_DIR), str(REPO_DIR / "compe")]
)

server_command = [
    str(PI05_PYTHON),
    "policy_server.py",
    "--port",
    str(SERVER_PORT),
]
eval_command = [
    str(EVAL_PYTHON),
    "-m",
    "pipeline",
    "--server-url",
    SERVER_URL,
    "--track",
    "track1",
    "--n-episodes",
    str(EVAL_EPISODES_PER_TASK),
    "--max-steps",
    str(EVAL_MAX_STEPS),
    "--seed",
    str(EVAL_SEED),
    "--timeout",
    "10",
    "--output-dir",
    str(RESULTS_DIR),
    *(["--record-video"] if RECORD_VIDEO else []),
    "--videos-per-task",
    str(VIDEOS_PER_TASK),
    "--video-fps",
    "20",
    "--tasks",
    *PUBLIC_TASK_IDS,
]

evaluation_process = None
with server_log_path.open("w", encoding="utf-8") as server_log:
    server_process = subprocess.Popen(
        server_command,
        cwd=REPO_DIR / "submission_template",
        env=server_env,
        stdout=server_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    try:
        deadline = time.monotonic() + 180
        while time.monotonic() < deadline:
            if server_process.poll() is not None:
                raise RuntimeError(f"policy server exited early; see {server_log_path}")
            try:
                with urllib.request.urlopen(f"{SERVER_URL}/health", timeout=2) as response:
                    if response.status == 200:
                        break
            except Exception:
                time.sleep(1)
        else:
            raise TimeoutError(f"policy server did not start; see {server_log_path}")

        print("π0.5 policy server ready. 公開4タスク評価を開始します。", flush=True)
        evaluation_process = subprocess.Popen(
            eval_command,
            cwd=REPO_DIR,
            env=eval_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        recent_eval_lines = []
        assert evaluation_process.stdout is not None
        with evaluation_log_path.open("w", encoding="utf-8") as evaluation_log:
            for line in evaluation_process.stdout:
                print(line, end="", flush=True)
                evaluation_log.write(line)
                evaluation_log.flush()
                recent_eval_lines.append(line.rstrip())
                if len(recent_eval_lines) > 120:
                    recent_eval_lines.pop(0)
        evaluation_returncode = evaluation_process.wait()
        if evaluation_returncode != 0:
            raise RuntimeError(
                f"evaluation failed with exit={evaluation_returncode}\n"
                + "\n".join(recent_eval_lines)
            )
    finally:
        if evaluation_process is not None and evaluation_process.poll() is None:
            evaluation_process.terminate()
            try:
                evaluation_process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                evaluation_process.kill()
                evaluation_process.wait(timeout=5)
        server_process.terminate()
        try:
            server_process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server_process.kill()
            server_process.wait(timeout=5)

if not public_eval_result_path.is_file():
    raise FileNotFoundError(public_eval_result_path)
public_eval_result = json.loads(public_eval_result_path.read_text(encoding="utf-8"))
track_result = public_eval_result["tracks"][0]
if track_result.get("overall_metrics", {}).get("error"):
    raise RuntimeError(f"track evaluation failed; see {public_eval_result_path}")
if len(track_result["tasks"]) != 4:
    raise RuntimeError(f"expected 4 task results, got {len(track_result['tasks'])}")

print("\n公開4タスク評価結果")
print(f"replan={REPLAN_STEPS}, inference_steps={INFERENCE_STEPS}")
print("task | success | steps | EE path | rotation | RMS jerk | SPARC | collision | sec")
print("--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---:")
for task in track_result["tasks"]:
    metrics = task["metrics"]
    print(
        f"{task['task_name']} | {task['success_rate']:.1%} | "
        f"{metrics.get('avg_steps_to_success', float('nan')):.1f} | "
        f"{metrics.get('cartesian_path_length', float('nan')):.3f} | "
        f"{metrics.get('orientation_path_length', float('nan')):.3f} | "
        f"{metrics.get('rms_cartesian_jerk', float('nan')):.3f} | "
        f"{metrics.get('sparc', float('nan')):.3f} | "
        f"{metrics.get('collision_rate', float('nan')):.1%} | "
        f"{metrics.get('avg_episode_time_sec', float('nan')):.1f}"
    )
print(f"overall success: {track_result['overall_score']:.1%}")
print("SPARCは0に近いほど滑らかです。成功率を保った候補間でsteps/path/rotation/jerkも比較します。")
print("result JSON:", public_eval_result_path)
print("server log:", server_log_path)
print("evaluation log:", evaluation_log_path)
video_paths = sorted((RESULTS_DIR / "videos").glob("*.mp4"))
print(f"videos: {len(video_paths)}")
for video_path in video_paths:
    print("  ", video_path)

DRIVE_EVAL_DIR = DRIVE_RUN_ROOT / "evaluation" / f"replan_{REPLAN_STEPS}"
DRIVE_EVAL_DIR.mkdir(parents=True, exist_ok=True)
for artifact in [public_eval_result_path, server_log_path, evaluation_log_path, *video_paths]:
    copy_file_with_progress(artifact, DRIVE_EVAL_DIR / artifact.name)
print("Drive evaluation artifacts:", DRIVE_EVAL_DIR)


### 評価動画を確認

失敗したエピソードだけを、左にagent view、右にwrist viewを並べたMP4で表示します。既定では各タスクの最初の失敗1件だけを保存します。全エピソードが成功したタスクには動画がありません。


In [ ]:
from IPython.display import Video, display

video_paths = sorted(
    (REPO_DIR / "results" / f"pi05_public_eval_replan_{REPLAN_STEPS}" / "videos").glob("*.mp4")
)
if not video_paths:
    print("失敗動画はありません（全成功、またはRECORD_VIDEO=False）。")
for video_path in video_paths:
    print(video_path.name)
    display(Video(str(video_path), embed=True, html_attributes="controls loop"))


In [ ]:
from google.colab import files

submission = REPO_DIR / "pi05_submission.zip"
if not submission.is_file():
    raise FileNotFoundError(submission)
print("submission size GiB:", round(submission.stat().st_size / 2**30, 2))
files.download(str(submission))
eval_result_candidates = sorted(
    (REPO_DIR / "results" / f"pi05_public_eval_replan_{REPLAN_STEPS}").glob("server_*.json")
)
if eval_result_candidates:
    files.download(str(eval_result_candidates[-1]))


## 提出前の最終確認

このノートブックでは静的検査、単体推論、公開4タスク評価を行います。公開評価では各HTTPリクエストの10秒制限、成功率、collision rateを確認できます。提出前には完成ZIPを本番相当環境でも `python validate_submission.py pi05_submission.zip` で確認し、必要に応じて20 episodes/taskで再評価してください。
